<a href="https://colab.research.google.com/github/bilalpervaiz/Geoai-climate-social-media-analysis/blob/main/GeoAI_and_climate_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GeoAI Analysis for climate-related social media data

This notebook demonstrates a **GeoAI (Geospatial Artificial Intelligence)** workflow to analyze climate-related social media activity in **Austria** between **2012 and 2023**.

### Key Objectives:
1.  **Exploratory Spatial Data Analysis (ESDA):** Visualizing the geographic distribution and temporal trends of climate-related posts across Austria.
2.  **Topic Modeling:** Using **BERTopic** to automatically discover the main themes within the climate discourse.
3.  **Emotion Analysis:** Applying a transformer-based classifier to detect human sentiments (joy, anger, fear, sadness) expressed in relation to climate topics.
4.  **Interactive Mapping:** Creating spatial visualizations to identify emotional and topical hotspots.

> ⚙️ Note:
If you're running this notebook in Google Colab, make sure to enable GPU acceleration for faster performance—especially when using transformer models (e.g. for topic or emotion classification).
You can do this by going to:
`Runtime` > `Change runtime type` > `Hardware accelerator: GPU (T4 GPU)`

To get started, let's install the required packages. Some packages like pandas or geopandas are already installed in Colab.

In [ ]:
# Install required Python libraries for the notebook:
# - contextily: for adding basemaps to maps in geopandas plots
# - folium: for interactive web-based maps
# - mapclassify: for classifying spatial data (e.g. choropleths)
# - bertopic: for topic modeling using transformer-based embeddings
# - wordcloud: for visualizing word frequency as word clouds
%pip install contextily folium mapclassify bertopic wordcloud

In [ ]:
# Regular expressions for text cleaning
import re

# Natural Language Toolkit (NLTK) for NLP utilities
import nltk
nltk.download('stopwords')         # Download stopword list for filtering common words
nltk.download('vader_lexicon')     # Download VADER sentiment lexicon for sentiment analysis

# Data handling
import pandas as pd                # For working with tabular data
import geopandas as gpd            # For geospatial data operations

# Mapping and plotting
import folium                      # For interactive webmaps
import contextily as ctx           # To add basemaps to geospatial plots
import matplotlib.pyplot as plt    # For creating static plots
import plotly.express as px        # For creating interactive plots
from wordcloud import WordCloud    # For generating wordclouds

# Geometric operations
from shapely import Point          # To create and manipulate point locations

# Topic modeling
from umap import UMAP              # Dimension reduction with UMAP
from hdbscan import HDBSCAN        # Clustering based on HDBSCAN
from bertopic import BERTopic      # Transformer-based topic modeling tool
from sentence_transformers import SentenceTransformer  # SentenceBERT for text embedding
from sklearn.feature_extraction.text import CountVectorizer  # For text vectorization (used by BERTopic)

# NLP resources
from nltk.corpus import stopwords  # Common stopwords in various languages
from nltk.sentiment.vader import SentimentIntensityAnalyzer  # Rule-based sentiment analysis tool based on VADER

# Progress bar for loops and pandas operations
from tqdm import tqdm
tqdm.pandas()                      # Enable tqdm progress bars for pandas through `.progress_apply()`

# Transformers (Hugging Face) which allows us to pre-trained models for for emotion analysis and other NLP tasks
from transformers import pipeline                          # High-level NLP pipeline (e.g. for zero-shot classification)
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # Classes for loading pretrained models

At IT:U, we constantly have social media crawlers running to obtain a large number of posts from all over the world. We have extracted posts from **Austria** that mention **climate topics** between **2012 and 2023**. Let's load this dataset for GeoAI analysis!

In [ ]:
# download data from the cloud
!wget "https://drive.google.com/uc?export=download&id=1GkJah8ZrOf5X3kNHzXDB_KUCFzfsK1XR" -O climate_tweets_austria_2012_2023_prepared.parquet

In [ ]:
# Load all data from the extracted file
# gdf is short for for geo-data-frame
gdf = gpd.read_parquet('climate_tweets_austria_2012_2023_prepared.parquet')

# Display the number of rows and the first 10 rows
print(f'Loaded {len(gdf)} tweets')
gdf.head()

We can already display the posts in an interactive map. That's a lot of information.

In [ ]:
gdf.explore(
    # Nicer basemap
    tiles="https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_4hrr_1_f5d8d8c2fa1f6e2aff2e35d6",
    attr="CartoDB",

    # Tooltip on hover — short preview, wrapped
    tooltip=True,
    tooltip_kwds=dict(
        style="""
            background-color: white;
            border: 1px solid #ccc;
            border-radius: 4px;
            font-size: 13px;
            max-width: 400px;
            white-space: normal;
            word-wrap: break-word;
        """
    ),

    # Popup on click — full text, also wrapped
    popup=True,
    popup_kwds=dict(
        max_width=400,
        style="""
            font-size: 13px;
            white-space: normal;
            word-wrap: break-word;
        """
    )
)

Besides looking at where people posted, it's also helpful to explore when they posted. In the next step, we'll plot the number of tweets per day to see how activity changed over time. This is part of our basic exploratory analysis and helps us understand the temporal dynamics of the dataset.

In [ ]:
tweets_per_day = gdf.groupby(gdf['date'].dt.to_period('M').dt.start_time).size().reset_index(name='count')

fig = px.line(
    tweets_per_day,
    x='date',
    y='count',
    title=f'Number of tweets climate-related tweets per month in Austria',
    labels={'date': 'Date', 'count': 'Tweet count'},
    template='plotly_white',
)

fig.show()

Awesome! Now let's see what information we can extract from our dataset.

## 1. Topic Modelling
Now that we've explored the basic structure of the tweet data, we can begin analysing **what people were talking about**. One powerful way to do this is through topic modelling.

Topic modelling automatically finds groups of texts that talk about similar things. One modern tool for this is BERTopic.

It works in a multi-step manner:
- **Step 1:** Understanding the meaning of words
BERTopic uses a smart language model (like ChatGPT) to understand what each tweet means, not just which words it contains. So it knows that: "flood" and "water everywhere" are similar even if the exact words are different
- **Step 2:** Grouping similar tweets together
It then puts tweets with similar meanings into groups.
- **Step 3:** Naming the topics For each group, it looks at the most important words and creates a topic label.

In [ ]:
# Initialize BERTopic with a custom vectorizer that removes common stopwords
# We also remove "@user" and "http" which are common in cleaned tweets
excluded_words = stopwords.words('english') + stopwords.words('german') + ['@user', 'http']

# Define the vectorizer to use these stopwords
vectorizer_model = CountVectorizer(stop_words=excluded_words)

# Configure HDBSCAN for clustering
hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=5, prediction_data=True)

# Configure dimension reduction model
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# Initialize BERTopic with the vectorizer and a fixed number of topics
topic_model = BERTopic(
    embedding_model=SentenceTransformer("distiluse-base-multilingual-cased-v2"),  # efficient multi-language embedding model
    vectorizer_model=vectorizer_model,  # topic representation model with excluded words
    hdbscan_model=hdbscan_model,  # the clustering algorithm to use,
    umap_model=umap_model,  # the dimension reduction algorithm to use
    nr_topics=10,  # fixed number of topics
    verbose=True
)

Next, we will fit the topic model on the cleaned tweet texts. This step will:
- Generate topic clusters
- Assign a topic ID to each tweet
- Create a summary representation for each topic

We'll also store the results in our GeoDataFrame and save it for later use.

In [ ]:
# Fit the model to the tweet texts and get topic probabilities
topics, probs = topic_model.fit_transform(gdf['text'])

# Get summary info for each topic (e.g. top words, size)
topic_info = topic_model.get_topic_info().set_index('Topic')
topic_info.loc[topic_info.index == -1, 'Name'] = 'Undefined (Noise)'

# Add topic labels back to the tweet data
gdf['topic_id'] = topics
gdf['topic'] = gdf['topic_id'].apply(lambda x: topic_info.loc[x]['Name'])

# Save the annotated data as GeoPackage file (optional)
gdf.to_parquet('climate_tweets_w_topics.gpkg')

# Display the topic overview
topic_info

Now let's visualize the top 10 discovered topics as a bar chart. This gives a quick overview of what people were tweeting about most in the flood-affected areas.

In [ ]:
# Visualize the 10 most common topics with their top words
topic_model.visualize_barchart(top_n_topics=10)

## 2. Explorative filtering with keywords
Now we know approximately what topics are present in our dataset. However, while topic modelling is great to get an overview of the data, it is not granular enough for precise filtering. For this purpose, it is better to use a targeted filtering mechanism to obtain relevant posts instead of only a topic modelling.

Going further, we want to examine how people are posting about the flood and climate-related topics specifically. To do that, we can use a simple **keyword filter** as below. Play around and see what data you can get out!

In [ ]:
KEYWORDS = ['Naturkatastrophen,]

# Filter for tweets containing keywords
filtered_gdf = gdf[
    gdf['text'].str.lower().str.contains('|'.join(KEYWORDS))
]
print(f'Found {len(filtered_gdf)} tweets containing keywords {KEYWORDS}')

# Display these tweets
filtered_gdf.head()

In [ ]:
filtered_gdf.explore(
    # Nicer basemap
    tiles="https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_4hrr_1_f5d8d8c2fa1f6e2aff2e35d6",
    attr="CartoDB",

    # Tooltip on hover — short preview, wrapped
    tooltip=True,
    tooltip_kwds=dict(
        style="""
            background-color: white;
            border: 1px solid #ccc;
            border-radius: 4px;
            font-size: 13px;
            max-width: 400px;
            white-space: normal;
            word-wrap: break-word;
        """
    ),

    # Popup on click — full text, also wrapped
    popup=True,
    popup_kwds=dict(
        max_width=400,
        style="""
            font-size: 13px;
            white-space: normal;
            word-wrap: break-word;
        """
    )
)

Again, let's also explore how people posted the filtered tweets over time.

In [ ]:
filtered_tweets_per_day = filtered_gdf.groupby(filtered_gdf['date'].dt.to_period('M').dt.start_time).size().reset_index(name='count')

fig = px.line(
    filtered_tweets_per_day,
    x='date',
    y='count',
    title=f'Number of tweets with keywords "{", ".join(KEYWORDS)}" per month (Austria)',
    labels={'date': 'Date', 'count': 'Tweet count'},
    template='plotly_white',
)

fig.show()

**Critical questions**:
- What limitations do you observe?
- What could be the use of such output?

## 2. Emotion Analysis
Now that we know which social media posts are related to the floods, we also would like to know what emotions people express online, to identify areas in most critical need.

To detect emotions, we use a pre-trained language model that is again similar to popular generative AI like ChatGPT. This particular model was trained for emotion detection on tweets across 19 languages.

In [ ]:
# Load the transformer-based emotion classifier
# This model returns scores for multiple emotion labels
classifier = pipeline(
    'text-classification',
    model="MilaNLProc/xlm-emo-t",  # Pretrained TweetNLP model
    return_all_scores=True,   # Return scores for all emotions
    max_length=512,           # Truncate long texts
    truncation=True
)

In [ ]:
def extract_emotions(text):
    # Get emotion predictions for a single tweet
    preds = classifier(text)

    # Extract the single most likely emotion
    single_label = max(preds, key=lambda x: x['score'])['label']

    # Extract all emotions with high confidence (> 0.5)
    multi_label = [pred['label'] for pred in preds if pred['score'] > 0.5]

    return single_label, multi_label

Emotions are really complex, and it's often hard to clearly identify an emotion just from text. It can also be the case that several emotions occur at once in the same text.

To handle this, we extract both the primary emotion (`emotion_label_single`) and the set of all present emotions (`emotion_multi_label`).

In [ ]:
# Apply emotion classification to each tweet and store both single and multi-labels
gdf[['emotion_single_label', 'emotion_multi_label']] = gdf['text'].progress_apply(
    lambda txt: pd.Series(extract_emotions(txt))
)

gdf['emotion_single_label'] = gdf.apply(
    lambda row: row['emotion_single_label'] if len(row['emotion_multi_label']) > 0 else 'none',
axis=1)

# Store the results to a file again
gdf.to_file('climate_tweets_w_topics_and_emotions.gpkg')

# Preview the results
gdf.head()

In [ ]:
emotion_counts = gdf['emotion_single_label'].value_counts().reset_index()
emotion_counts.columns = ['emotion', 'count']

fig = px.bar(
    emotion_counts,
    x='emotion',
    y='count',
    title='Emotion distribution within all climate-related tweets in Austria (2012-2023)',
    labels={'emotion': 'Emotion', 'count': 'Count'},
    template='plotly_white',
)

fig.show()

We can see that a lot of the disaster-related posts seem to contain joyful emotions. Why would that be? Let's have a look at the data on the map.

In [ ]:
emotion_colors = {
    "joy":     "#2ecc71",  # green
    "fear":    "#9b59b6",  # purple
    "anger":   "#e74c3c",  # red
    "none":    "#bdc3c7",  # gray
    "sadness": "#3498db",  # blue
}

gdf.explore(
    column='emotion_single_label',
    cmap=['#e74c3c', '#9b59b6', '#2ecc71', '#bdc3c7', '#3498db'],  # anger, fear, joy, none, sadness
    categories=['anger', 'fear', 'joy', 'none', 'sadness'],         # explicit order to be safe

    tiles="https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_4hrr_1_f5d8d8c2fa1f6e2aff2e35d6",
    attr="CartoDB",
    tooltip=True,
    tooltip_kwds=dict(
        style="""
            background-color: white;
            border: 1px solid #ccc;
            border-radius: 4px;
            font-size: 13px;
            max-width: 400px;
            white-space: normal;
            word-wrap: break-word;
        """
    ),
    popup=True,
    popup_kwds=dict(
        max_width=400,
        style="""
            font-size: 13px;
            white-space: normal;
            word-wrap: break-word;
        """
    )
)

Finally, it would be interesting to see how emotions proceed over time.

In [ ]:
# Resample to weekly counts
emotion_tweets_per_week = (
    gdf.groupby([gdf['date'].dt.to_period('M').dt.start_time, 'emotion_single_label'])
    .size()
    .reset_index(name='count')
)

fig = px.line(
    emotion_tweets_per_week,
    x='date',
    y='count',
    color='emotion_single_label',
    title='Emotions of climate-related tweets per month in Austria',
    color_discrete_map=emotion_colors,
    labels={'date': 'Date', 'count': 'Tweet count', 'emotion_single_label': 'Emotion'},
    template='plotly_white'
)

fig.update_traces(line=dict(width=1.5))
fig.update_layout(legend=dict(title='Emotion'))

fig.show()

**Critical questions:**
- What patterns can you observe?
- How reliable do you think the emotion detection is?